In [1]:
# traffic_app.py
import os
import csv
import json
import time
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
from flask import Flask, jsonify, request, send_from_directory
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, hour as spark_hour
import findspark
findspark.init()

In [2]:
# ------------------------------
# 1. CẤU HÌNH
# ------------------------------

TOMTOM_API_KEY = "J2KZlTgfK0r03EB2wUjhe3ZMxgDCXMoT"   
GOOGLE_API_KEY = ""   

# Danh sách tuyến đường cần theo dõi (tọa độ thực tế tại TP.HCM)
ROUTES = [
    {"id": "A->B", "origin": "10.7769,106.7009", "destination": "10.8231,106.6296"},
    {"id": "B->C", "origin": "10.8231,106.6296", "destination": "10.8012,106.6678"},
    {"id": "C->D", "origin": "10.8012,106.6678", "destination": "10.7512,106.6943"},
    {"id": "A->C", "origin": "10.7769,106.7009", "destination": "10.8012,106.6678"},
    {"id": "B->D", "origin": "10.8231,106.6296", "destination": "10.7512,106.6943"},
]

In [3]:
# ------------------------------
# 2. HÀM TẠO DỮ LIỆU MẪU (FALLBACK)
# ------------------------------
def generate_mock_data(num_records=20000, save_path='data/traffic_data.csv'):
    os.makedirs('data', exist_ok=True)
    if os.path.exists(save_path):
        print("Dữ liệu mẫu đã tồn tại, bỏ qua tạo mới.")
        return
    routes = [r['id'] for r in ROUTES]
    base_time = datetime(2025, 4, 1, 0, 0, 0)
    data = []
    for i in range(num_records):
        route = np.random.choice(routes)
        hour = np.random.randint(0, 24)
        minute = np.random.randint(0, 60)
        timestamp = base_time + timedelta(days=np.random.randint(0, 7), hours=hour, minutes=minute)
        # Mô phỏng tốc độ theo giờ
        if 7 <= hour <= 9 or 17 <= hour <= 19:
            speed = np.random.uniform(15, 35)
        elif 22 <= hour or hour <= 5:
            speed = np.random.uniform(50, 80)
        else:
            speed = np.random.uniform(30, 60)
        volume = np.random.poisson(lam=100) * (1.5 if (7<=hour<=9 or 17<=hour<=19) else 0.7)
        travel_time = 10 / (speed / 60)
        data.append([route, timestamp, round(speed,1), int(volume), round(travel_time,1)])
    df = pd.DataFrame(data, columns=['route_id', 'timestamp', 'speed_kmh', 'vehicle_volume', 'travel_time_min'])
    df.to_csv(save_path, index=False)
    print(f"Đã tạo {num_records} bản ghi mẫu tại {save_path}")

In [4]:
# ==================== THIẾT LẬP MONGODB ====================
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "traffic_db"
COLLECTION_NAME = "traffic_records"
STATS_COLLECTION = "route_stats"

def get_mongo_client():
    return MongoClient(MONGO_URI)

def get_db():
    return get_mongo_client()[DB_NAME]

def init_mongo():
    db = get_db()
    db[COLLECTION_NAME].create_index([("route_id", 1), ("timestamp", 1)])
    db[STATS_COLLECTION].create_index([("route_id", 1), ("hour", 1)])
    print("🔌 MongoDB connected & indexed")

In [5]:
# ------------------------------
# 3. Lấy dữ liệu từ API TomTom, Google, mô phỏng
# ------------------------------
def fetch_tomtom_traffic(point):
    if not TOMTOM_API_KEY:
        return None
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {
        "key": TOMTOM_API_KEY,
        "point": point,
        "unit": "kmph"
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get('flowSegmentData'):
                flow = data['flowSegmentData']
                current_speed = flow.get('currentSpeed', 0)
                free_flow_speed = flow.get('freeFlowSpeed', 0)
                confidence = flow.get('confidence', 0)
                if current_speed == 0 and free_flow_speed > 0:
                    current_speed = free_flow_speed * 0.7
                return {
                    "speed_kmh": current_speed,
                    "free_flow_speed": free_flow_speed,
                    "confidence": confidence
                }
    except Exception as e:
        print(f"TomTom Traffic error: {e}")
    return None

def fetch_tomtom_route(origin, dest):
    if not TOMTOM_API_KEY:
        return None
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{origin}:{dest}/json"
    params = {"key": TOMTOM_API_KEY, "routeType": "fastest", "travelMode": "car", "traffic": "true"}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get('routes'):
                s = data['routes'][0]['summary']
                travel_min = s['travelTimeInSeconds'] / 60
                dist_km = s['lengthInMeters'] / 1000
                speed = (dist_km / (travel_min/60)) if travel_min>0 else 0
                return {"travel_time_min": round(travel_min,1), "speed_kmh": round(speed,1), "distance_km": round(dist_km,1)}
    except Exception as e:
        print(f"TomTom Routing error: {e}")
    return None

def fetch_simulated_realtime(route_id):
    hour = datetime.now().hour
    if 7 <= hour <= 9 or 17 <= hour <= 19:
        speed = np.random.uniform(15, 35)
    else:
        speed = np.random.uniform(40, 70)
    travel_time = 10 / (speed / 60)
    return {"travel_time_min": round(travel_time,1), "speed_kmh": round(speed,1), "source": "Simulated"}

def log_search(origin_addr, dest_addr, origin_coord, dest_coord, routes):
    """Ghi log mỗi lần tìm đường vào file CSV"""
    log_file = 'data/search_log.csv'
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['timestamp', 'origin_address', 'dest_address', 'origin_lat', 'origin_lon', 'dest_lat', 'dest_lon', 'routes_summary'])
        routes_summary = ' | '.join([f"T{i+1}:{r['time']}min-{r['distance']}km" for i, r in enumerate(routes)])
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            origin_addr,
            dest_addr,
            origin_coord['lat'],
            origin_coord['lon'],
            dest_coord['lat'],
            dest_coord['lon'],
            routes_summary
        ])

def update_data_with_realtime(data_file='data/traffic_data.csv'):
    print("🔄 Đang cập nhật dữ liệu thời gian thực từ TomTom...")
    new_records = []
    for route in ROUTES:
        traffic_data = fetch_tomtom_traffic(route['origin'])
        if traffic_data and traffic_data['speed_kmh'] > 0:
            speed = traffic_data['speed_kmh']
            travel_time = 10 / (speed / 60)
            source = "TomTom Traffic"
        else:
            route_data = fetch_tomtom_route(route['origin'], route['destination'])
            if route_data:
                speed = route_data['speed_kmh']
                travel_time = route_data['travel_time_min']
                source = "TomTom Routing"
            else:
                sim = fetch_simulated_realtime(route['id'])
                speed = sim['speed_kmh']
                travel_time = sim['travel_time_min']
                source = sim['source']
        record = {
            'route_id': route['id'],
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'speed_kmh': speed,
            'vehicle_volume': np.random.randint(50, 200),
            'travel_time_min': travel_time
        }
        new_records.append(record)
        print(f"  ✅ Route {route['id']}: {travel_time} phút, {speed} km/h ({source})")
        time.sleep(0.5)
    
    # ---- LƯU VÀO MONGODB ----
    db = get_db()
    collection = db[COLLECTION_NAME]
    try:
        mongo_records = []
        for rec in new_records:
            r = rec.copy()
            r['timestamp'] = datetime.strptime(r['timestamp'], "%Y-%m-%d %H:%M:%S")
            mongo_records.append(r)
        collection.insert_many(mongo_records)
        print(f"🍃 Đã thêm {len(mongo_records)} bản ghi vào MongoDB")
    except Exception as e:
        print(f"Lỗi MongoDB: {e}")
    
    # ---- GHI CSV DỰ PHÒNG ----
    os.makedirs('data', exist_ok=True)
    df_new = pd.DataFrame(new_records)
    if os.path.exists(data_file):
        df_old = pd.read_csv(data_file)
        if 'timestamp' in df_old.columns:
            df_old['timestamp'] = pd.to_datetime(df_old['timestamp'], errors='coerce')
            df_old = df_old.dropna(subset=['timestamp'])
            df_old['timestamp'] = df_old['timestamp'].dt.strftime("%Y-%m-%d %H:%M:%S")
        df_combined = pd.concat([df_old, df_new], ignore_index=True)
        df_combined['timestamp'] = pd.to_datetime(df_combined['timestamp'])
        df_combined = df_combined.sort_values('timestamp').tail(5000)
        df_combined['timestamp'] = df_combined['timestamp'].dt.strftime("%Y-%m-%d %H:%M:%S")
    else:
        df_combined = df_new
    df_combined.to_csv(data_file, index=False)
    print(f"💾 Đã cập nhật {len(new_records)} bản ghi vào {data_file}")
    return df_combined

In [6]:
# ------------------------------
# 4. XỬ LÝ SPARK
# ------------------------------
def init_spark():
    return SparkSession.builder \
        .appName("TrafficRecommender") \
        .config("spark.mongodb.read.connection.uri", MONGO_URI) \
        .config("spark.mongodb.write.connection.uri", MONGO_URI) \
        .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.2.1") \
        .getOrCreate()

def run_spark_pipeline(use_mongo=True, out_path='data/route_stats.csv'):
    route_distances = {}
    from math import radians, sin, cos, sqrt, atan2
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        dlat = radians(lat2 - lat1)
        dlon = radians(lon2 - lon1)
        a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        return R * c
    for route in ROUTES:
        lat1, lon1 = map(float, route['origin'].split(','))
        lat2, lon2 = map(float, route['destination'].split(','))
        route_distances[route['id']] = round(haversine(lat1, lon1, lat2, lon2), 1)

    if not use_mongo:
        # Fallback: đọc CSV
        if not os.path.exists('data/traffic_data.csv'):
            generate_mock_data(20000)
        spark = init_spark()
        df = spark.read.csv('data/traffic_data.csv', header=True, inferSchema=True)
        # Xử lý như cũ ...
    else:
        # Đọc từ MongoDB
        spark = init_spark()
        df = spark.read.format("mongodb") \
            .option("database", DB_NAME) \
            .option("collection", COLLECTION_NAME) \
            .load()
    
    # Làm sạch chung
    df_clean = df.dropna(subset=['speed_kmh', 'travel_time_min'])
    df_clean = df_clean.filter((col('speed_kmh') <= 120) & (col('speed_kmh') >= 5))
    df_clean = df_clean.filter(col('travel_time_min') >= 1)
    df_clean = df_clean.withColumn('hour', spark_hour(col('timestamp')))
    
    stats = df_clean.groupBy('route_id', 'hour').agg(
        avg('speed_kmh').alias('avg_speed'),
        avg('travel_time_min').alias('avg_travel_time')
    )
    stats_pd = stats.toPandas()
    stats_pd['distance_km'] = stats_pd['route_id'].map(route_distances)
    
    # Lưu vào MongoDB stats collection và CSV
    if use_mongo:
        stats_coll = get_db()[STATS_COLLECTION]
        stats_coll.delete_many({})
        stats_coll.insert_many(stats_pd.to_dict('records'))
        print("📊 Đã lưu thống kê tuyến vào MongoDB")
    stats_pd.to_csv(out_path, index=False)
    spark.stop()
    return stats_pd

def get_recommendations(use_mongo=True, csv_path='data/route_stats.csv'):
    if use_mongo:
        stats_coll = get_db()[STATS_COLLECTION]
        docs = list(stats_coll.find({}, {'_id': 0}))
        if docs:
            stats = pd.DataFrame(docs)
        else:
            stats = pd.DataFrame(columns=['route_id', 'hour', 'avg_travel_time', 'distance_km'])
    else:
        if not os.path.exists(csv_path):
            run_spark_pipeline(use_mongo=False)
        stats = pd.read_csv(csv_path)
    
    recommendations = {}
    for hour in range(24):
        hour_stats = stats[stats['hour'] == hour].sort_values('avg_travel_time').head(5)
        rec_list = [{"route": r['route_id'], "time": round(r['avg_travel_time'],1), "distance": r['distance_km']} for _, r in hour_stats.iterrows()]
        recommendations[hour] = rec_list
    return recommendations

In [7]:
# ------------------------------
# 5. ĐÁNH GIÁ
# ------------------------------
def evaluate_model(data_path='data/traffic_data.csv', stats_path='data/route_stats.csv'):
    import numpy as np
    from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
    if not os.path.exists(data_path) or not os.path.exists(stats_path):
        print("⚠️ Chưa có đủ dữ liệu để đánh giá.")
        return None
    actual = pd.read_csv(data_path)
    if 'timestamp' not in actual.columns:
        return None
    actual['timestamp'] = pd.to_datetime(actual['timestamp'], errors='coerce')
    actual = actual.dropna(subset=['timestamp'])
    if actual.empty:
        return None
    actual['hour'] = actual['timestamp'].dt.hour
    actual_mean = actual.groupby(['route_id','hour'])['travel_time_min'].mean().reset_index()
    pred = pd.read_csv(stats_path)
    merged = pd.merge(actual_mean, pred, on=['route_id','hour'], how='inner')
    if merged.empty:
        return None
    y_true = merged['travel_time_min']
    y_pred = merged['avg_travel_time']
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    # Baseline: dự đoán bằng trung bình toàn bộ
    global_mean = y_true.mean()
    baseline_mae = mean_absolute_error(y_true, [global_mean]*len(y_true))
    improvement_pct = (baseline_mae - mae) / baseline_mae * 100
    
    results = {
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "mape": round(mape, 2),
        "baseline_mae": round(baseline_mae, 2),
        "improvement_%": round(improvement_pct, 2)
    }
    print(f"📊 Đánh giá: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.1f}%")
    print(f"   So với baseline (MAE={baseline_mae:.2f}): cải thiện {improvement_pct:.1f}%")
    return results

In [8]:
# ==================== TÍCH HỢP R (thống kê) ====================
R_AVAILABLE = False
try:
    import rpy2.robjects as robjects
    from rpy2.robjects import vectors
    from rpy2.robjects.conversion import localconverter
    # Không import pandas2ri
    R_AVAILABLE = True
    print("✅ R connected")
except ImportError:
    print("⚠️ chưa cài rpy2, chức năng phân tích R sẽ bị tắt")

def run_r_analysis(data_path='data/traffic_data.csv'):
    if not R_AVAILABLE:
        return {"error": "R (rpy2) is not installed"}
    if not os.path.exists(data_path):
        return {"error": f"CSV file {data_path} not found"}
    try:
        df = pd.read_csv(data_path)
        if 'timestamp' not in df.columns:
            return {"error": "No timestamp column"}
        df['hour'] = pd.to_datetime(df['timestamp']).dt.hour

        # Sử dụng context converter để tránh lỗi threading
        with localconverter(robjects.default_converter):
            r_travel = vectors.FloatVector(df['travel_time_min'].tolist())
            r_hour = vectors.IntVector(df['hour'].tolist())
            r_route = vectors.StrVector(df['route_id'].tolist())

            robjects.globalenv['r_travel'] = r_travel
            robjects.globalenv['r_hour'] = r_hour
            robjects.globalenv['r_route'] = r_route

            robjects.r('''
                traffic_data <- data.frame(
                    travel_time_min = r_travel,
                    hour = r_hour,
                    route_id = r_route,
                    stringsAsFactors = TRUE
                )
            ''')

            robjects.r('''
                tryCatch({
                    model <- lm(travel_time_min ~ factor(hour) + route_id, data = traffic_data)
                    summ <- summary(model)
                    r2 <- summ$r.squared
                    fstat <- summ$fstatistic
                    p_value <- pf(fstat[1], fstat[2], fstat[3], lower.tail=FALSE)
                    cat(paste("R-squared:", round(r2,4),
                              "\\nF-statistic:", round(fstat[1],2),
                              "\\nP-value:", format.pval(p_value, digits=4)))
                }, error = function(e) {
                    cat("R error:", e$message)
                    r2 <- NA
                })
            ''')

            r2_val = robjects.globalenv['r2'][0]
            if not np.isfinite(r2_val):
                return {"error": "R model failed"}

            model = robjects.globalenv['model']
            coef = robjects.r['coefficients'](model)
            coef_dict = {str(k): round(float(v), 4) for k, v in zip(coef.names, list(coef))}

        return {
            "r_squared": round(r2_val, 4),
            "coefficients": coef_dict,
            "note": "Model: travel_time_min ~ hour + route_id"
        }
    except Exception as e:
        return {"error": f"R analysis error: {str(e)}"}

# ==================== DATA REPAIR ====================
def repair_data(data_path='data/traffic_data.csv'):
    """
    Phát hiện và sửa các bản ghi lỗi đơn giản.
    """
    if not os.path.exists(data_path):
        return None
    df = pd.read_csv(data_path)
    fixed = 0
    for idx, row in df.iterrows():
        if row['travel_time_min'] <= 0 and row['speed_kmh'] > 0:
            df.at[idx, 'travel_time_min'] = 10 / (row['speed_kmh'] / 60)
            fixed += 1
        if row['speed_kmh'] <= 0:
            df.at[idx, 'speed_kmh'] = np.nan
            fixed += 1
    if fixed > 0:
        df.to_csv(data_path, index=False)
        # Đồng bộ MongoDB nếu kết nối được
        try:
            db = get_db()
            db[COLLECTION_NAME].delete_many({"speed_kmh": {"$lte": 0}})
            print(f"🔧 Sửa {fixed} bản ghi lỗi, đã xóa bản ghi lỗi trên MongoDB")
        except:
            print(f"🔧 Sửa {fixed} bản ghi lỗi trong CSV (MongoDB không khả dụng)")
    return df

# ==================== EXPERIMENT ====================
def run_experiments(iterations=5):
    """
    Chạy thí nghiệm nhiều lần: thu thập dữ liệu mới, tính toán Spark, đánh giá.
    Lưu log vào data/experiment_log.csv
    """
    log = []
    for i in range(iterations):
        try:
            update_data_with_realtime()          # lấy dữ liệu thời gian thực (tự động ghi CSV + MongoDB)
            run_spark_pipeline(use_mongo=True)   # tính toán từ MongoDB
            res = evaluate_model()               # đánh giá
            if res:
                res['iteration'] = i + 1
                log.append(res)
        except Exception as e:
            print(f"Lỗi ở lần lặp {i+1}: {e}")
        time.sleep(1)
    if log:
        pd.DataFrame(log).to_csv('data/experiment_log.csv', index=False)
        print("🧪 Hoàn thành thí nghiệm, log lưu tại data/experiment_log.csv")
    return log
# ==================== TẠO HEATMAP ====================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

def generate_heatmap(stats_path='data/route_stats.csv', output_path='frontend/static/heatmap.png'):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    if not os.path.exists(stats_path):
        run_spark_pipeline(use_mongo=False)
    stats = pd.read_csv(stats_path)
    pivot = stats.pivot_table(values='avg_travel_time', index='route_id', columns='hour', aggfunc='mean')
    plt.figure(figsize=(14, 6))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Thời gian (phút)'})
    plt.title('Thời gian di chuyển trung bình theo giờ và tuyến đường')
    plt.xlabel('Giờ trong ngày')
    plt.ylabel('Tuyến đường')
    plt.tight_layout()
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"✅ Heatmap đã lưu tại {output_path}")

✅ R connected


In [ ]:
# ------------------------------
# FLASK APP & API
# ------------------------------
app = Flask(__name__)

# CRUD helpers cho data_management (dùng CSV - giữ nguyên để tương thích trang quản lý)
CSV_FILE = 'data/traffic_data.csv'
def get_all_records():
    if not os.path.exists(CSV_FILE): return []
    return pd.read_csv(CSV_FILE).to_dict('records')
def add_record(record):
    df = pd.read_csv(CSV_FILE)
    df = pd.concat([df, pd.DataFrame([record])], ignore_index=True)
    df.to_csv(CSV_FILE, index=False)
def update_record(old_route, old_ts, new_rec):
    df = pd.read_csv(CSV_FILE)
    mask = (df['route_id']==old_route) & (df['timestamp']==old_ts)
    for k,v in new_rec.items(): df.loc[mask,k]=v
    df.to_csv(CSV_FILE, index=False)
def delete_record(route_id, ts):
    df = pd.read_csv(CSV_FILE)
    df = df[~((df['route_id']==route_id) & (df['timestamp']==ts))]
    df.to_csv(CSV_FILE, index=False)
def log_search(origin_addr, dest_addr, origin_coord, dest_coord, routes):
    """Ghi log mỗi lần tìm đường vào file CSV"""
    log_file = 'data/search_log.csv'
    file_exists = os.path.isfile(log_file)
    with open(log_file, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['timestamp', 'origin_address', 'dest_address', 'origin_lat', 'origin_lon', 'dest_lat', 'dest_lon', 'routes_summary'])
        routes_summary = ' | '.join([f"T{i+1}:{r['time']}min-{r['distance']}km" for i, r in enumerate(routes)])
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            origin_addr,
            dest_addr,
            origin_coord['lat'],
            origin_coord['lon'],
            dest_coord['lat'],
            dest_coord['lon'],
            routes_summary
        ])

@app.route('/api/routes-between')
def routes_between():
    origin = request.args.get('origin')
    dest = request.args.get('dest')
    origin_addr = request.args.get('origin_addr', '')
    dest_addr = request.args.get('dest_addr', '')
    if not origin or not dest:
        return jsonify({'error': 'Missing origin/dest'}), 400
    if not TOMTOM_API_KEY:
        return jsonify({'error': 'No TomTom API key'}), 400
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{origin}:{dest}/json"
    params = {
        "key": TOMTOM_API_KEY,
        "routeType": "fastest",
        "travelMode": "car",
        "traffic": "true",
        "maxAlternatives": 3,
        "instructions": "false"
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code == 200:
            data = r.json()
            routes = data.get('routes', [])
            result = []
            for idx, route in enumerate(routes):
                summary = route['summary']
                travel_min = summary['travelTimeInSeconds'] / 60
                distance_km = summary['lengthInMeters'] / 1000
                points = route['legs'][0]['points']
                coords = [[p['latitude'], p['longitude']] for p in points]
                result.append({
                    "index": idx,
                    "time": round(travel_min, 1),
                    "distance": round(distance_km, 1),
                    "coordinates": coords
                })
            # GHI LOG nếu có địa chỉ
            if origin_addr and dest_addr:
                try:
                    o_lat, o_lon = map(float, origin.split(','))
                    d_lat, d_lon = map(float, dest.split(','))
                    log_search(origin_addr, dest_addr, {'lat': o_lat, 'lon': o_lon}, {'lat': d_lat, 'lon': d_lon}, result)
                except Exception as e:
                    print(f"Lỗi ghi log: {e}")
            return jsonify(result)
        else:
            return jsonify({'error': f'TomTom API error: {r.status_code}'}), 500
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/generate-heatmap')
def generate_heatmap():
    try:
        # Đọc dữ liệu thống kê tuyến đường
        stats_path = 'data/route_stats.csv'
        if not os.path.exists(stats_path):
            # Nếu chưa có file stats, tạm thời tạo từ MongoDB hoặc CSV gốc
            run_spark_pipeline(use_mongo=False)  # hoặc use_mongo=True nếu có
        df = pd.read_csv(stats_path)
        if df.empty:
            return jsonify({'error': 'No data'}), 400
        
        # Tạo bảng pivot (hàng: route_id, cột: hour, giá trị: avg_travel_time)
        pivot = df.pivot(index='route_id', columns='hour', values='avg_travel_time')
        
        # Vẽ heatmap
        plt.figure(figsize=(12, 8))
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.5)
        plt.title('Thời gian di chuyển trung bình (phút) theo giờ và tuyến đường')
        plt.xlabel('Giờ trong ngày')
        plt.ylabel('Tuyến đường')
        plt.tight_layout()
        
        # Lưu vào thư mục static
        static_dir = 'frontend/static'
        os.makedirs(static_dir, exist_ok=True)
        heatmap_path = os.path.join(static_dir, 'heatmap.png')
        plt.savefig(heatmap_path, dpi=150)
        plt.close()
        
        return jsonify({'status': 'ok', 'message': 'Heatmap created', 'path': '/static/heatmap.png'})
    except Exception as e:
        print(f"Error generating heatmap: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/api/geocode')
def geocode():
    address = request.args.get('address')
    if not address:
        return jsonify({'error': 'Missing address'}), 400
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1, "countrycodes": "vn"}
    try:
        time.sleep(1)
        r = requests.get(url, params=params, headers={"User-Agent": "TrafficApp/1.0"})
        if r.status_code == 200 and r.json():
            data = r.json()[0]
            return jsonify({'lat': float(data['lat']), 'lon': float(data['lon']), 'display_name': data['display_name']})
    except Exception as e:
        print(f"Geocode error: {e}")
    return jsonify({'error': 'Geocoding failed'}), 500

@app.route('/api/autocomplete')
def autocomplete():
    query = request.args.get('query')
    if not query or len(query) < 2:
        return jsonify([])
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": query, "format": "json", "limit": 5, "countrycodes": "vn"}
    try:
        time.sleep(1)
        r = requests.get(url, params=params, headers={"User-Agent": "TrafficApp/1.0"})
        if r.status_code == 200:
            results = [{'display_name': item['display_name'], 'lat': float(item['lat']), 'lon': float(item['lon'])} for item in r.json()]
            return jsonify(results)
    except Exception as e:
        print(f"Autocomplete error: {e}")
    return jsonify([])

@app.route('/api/reverse-geocode')
def reverse_geocode():
    lat = request.args.get('lat')
    lon = request.args.get('lon')
    if not lat or not lon:
        return jsonify({'error': 'Missing lat/lon'}), 400
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {"lat": lat, "lon": lon, "format": "json", "zoom": 18, "addressdetails": 1}
    try:
        time.sleep(1)
        r = requests.get(url, params=params, headers={"User-Agent": "TrafficApp/1.0"})
        if r.status_code == 200:
            data = r.json()
            if data.get('display_name'):
                return jsonify({'address': data['display_name']})
    except Exception as e:
        print(f"Reverse geocode error: {e}")
    return jsonify({'error': 'Reverse geocoding failed'}), 500

@app.route('/api/route-geometry')
def route_geometry():
    origin = request.args.get('origin')
    dest = request.args.get('dest')
    if not origin or not dest:
        return jsonify({'error': 'Missing origin/dest'}), 400
    if not TOMTOM_API_KEY:
        return jsonify({'error': 'No TomTom API key'}), 400
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{origin}:{dest}/json"
    params = {"key": TOMTOM_API_KEY, "routeType": "fastest", "travelMode": "car", "traffic": "true"}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            points = data['routes'][0]['legs'][0]['points']
            coords = [[p['latitude'], p['longitude']] for p in points]
            return jsonify({'coordinates': coords})
    except Exception as e:
        print(f"Route geometry error: {e}")
    return jsonify({'error': 'Cannot fetch route'}), 500

@app.route('/api/search-log')
def get_search_log():
    log_file = 'data/search_log.csv'
    if not os.path.exists(log_file):
        return jsonify([])
    with open(log_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        logs = list(reader)
    return jsonify(logs)

@app.route('/api/clear-search-log', methods=['POST'])
def clear_search_log():
    log_file = 'data/search_log.csv'
    if os.path.exists(log_file):
        with open(log_file, 'w', encoding='utf-8') as f:
            f.write('timestamp,origin_address,dest_address,origin_lat,origin_lon,dest_lat,dest_lon,routes_summary\n')
    return jsonify({'status': 'ok'})

@app.route('/api/recommendations')
def api_recommendations():
    hour = int(request.args.get('hour', 7))
    recs = get_recommendations(use_mongo=True).get(hour, [])
    return jsonify(recs)

@app.route('/api/experiment', methods=['POST'])
def api_experiment():
    its = int(request.args.get('iterations', 3))
    log = run_experiments(its)
    return jsonify(log)

@app.route('/api/fetch_now')
def api_fetch_now():
    update_data_with_realtime()
    run_spark_pipeline(use_mongo=True)
    return jsonify({'status': 'ok'})

@app.route('/api/report')
def api_report():
    result = evaluate_model()
    return jsonify(result if result else {})

# CRUD endpoints
@app.route('/api/data')
def api_data():
    return jsonify(get_all_records())

@app.route('/api/add', methods=['POST'])
def api_add():
    add_record(request.json)
    run_spark_pipeline()
    return jsonify({'status': 'ok'})

@app.route('/api/update', methods=['POST'])
def api_update():
    data = request.json
    update_record(data['old_route'], data['old_timestamp'], data['new_record'])
    run_spark_pipeline()
    return jsonify({'status': 'ok'})

@app.route('/api/delete', methods=['POST'])
def api_delete():
    data = request.json
    delete_record(data['route_id'], data['timestamp'])
    run_spark_pipeline()
    return jsonify({'status': 'ok'})

@app.route('/api/r_analysis')
def api_r_analysis():
    try:
        result = run_r_analysis()
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/repair', methods=['POST'])
def api_repair():
    repair_data()
    run_spark_pipeline(use_mongo=True)
    return jsonify({'status': 'repaired and repipeline'})

# Serve frontend
@app.route('/')
def index():
    return send_from_directory('frontend', 'index.html')

@app.route('/data_management')
def data_management():
    return send_from_directory('frontend', 'data_management.html')

@app.route('/analysis')
def analysis():
    return send_from_directory('frontend', 'analysis.html')

@app.route('/frontend/<path:filename>')
def serve_frontend(filename):
    return send_from_directory('frontend', filename)

# ========== THÊM ROUTE TĨNH ==========
@app.route('/static/<path:filename>')
def serve_static(filename):
    return send_from_directory('frontend/static', filename)


In [ ]:
# ------------------------------
# KHỞI ĐỘNG
# ------------------------------
if __name__ == '__main__':
    # --- 1. Kết nối MongoDB (nếu có) ---
    try:
        init_mongo()
        print("✅ MongoDB đã sẵn sàng")
    except Exception as e:
        print(f"⚠️ Không kết nối được MongoDB: {e}")
        print("   Hệ thống sẽ chỉ dùng file CSV.")

    # Tạo thư mục data nếu chưa có
    os.makedirs('data', exist_ok=True)

    # --- 2. Tạo dữ liệu mẫu nếu chưa có CSV ---
    if not os.path.exists('data/traffic_data.csv'):
        generate_mock_data(20000)

    # --- 3. Tạo thống kê ban đầu (dùng CSV, tránh MongoDB nếu chưa sẵn sàng) ---
    if not os.path.exists('data/route_stats.csv'):
        try:
            run_spark_pipeline(use_mongo=False)   # tạo từ CSV
        except Exception as e:
            print(f"❌ Lỗi Spark pipeline (tạo stats): {e}")
            # Tạo file stats trống để server không bị crash
            pd.DataFrame(columns=['route_id','hour','avg_speed','avg_travel_time','distance_km'])\
              .to_csv('data/route_stats.csv', index=False)

    # --- 4. Đánh giá ban đầu ---
    try:
        evaluate_model()
    except Exception as e:
        print(f"⚠️ Không đánh giá được mô hình ban đầu: {e}")

    # --- 5. Khởi động server ---
    print("\n🚀 Web server chạy tại http://localhost:5000")
    app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)

🔌 MongoDB connected & indexed
✅ MongoDB đã sẵn sàng
📊 Đánh giá: MAE=0.28, RMSE=1.05, MAPE=1.5%
   So với baseline (MAE=5.43): cải thiện 94.9%

🚀 Web server chạy tại http://localhost:5000
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.122:5000
Press CTRL+C to quit
127.0.0.1 - - [29/Apr/2026 21:07:49] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [29/Apr/2026 21:07:59] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [29/Apr/2026 21:08:10] "GET /data_management HTTP/1.1" 304 -
127.0.0.1 - - [29/Apr/2026 21:08:11] "GET /api/search-log HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:11] "GET /api/data HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:11] "GET /api/search-log HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:19] "GET /api/search-log HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:46] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [29/Apr/2026 21:08:54] "GET /api/autocomplete?query=121 HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:56] "GET /api/autocomplete?query=121%20Lê%20ĐIn HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:08:58] "GET /api/autocomplete?query=121%20Lê%20Đình%20Cẩn HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:09:

📊 Đánh giá: MAE=0.28, RMSE=1.05, MAPE=1.5%
   So với baseline (MAE=5.43): cải thiện 94.9%
R-squared: 0.5968 
F-statistic: 272.58 
P-value: < 2.2e-16

127.0.0.1 - - [29/Apr/2026 21:14:16] "GET /api/r_analysis HTTP/1.1" 200 -


🔄 Đang cập nhật dữ liệu thời gian thực từ TomTom...
  ✅ Route A->B: 40.0 phút, 15 km/h (TomTom Traffic)
  ✅ Route B->C: 23.076923076923077 phút, 26 km/h (TomTom Traffic)
  ✅ Route C->D: 26.08695652173913 phút, 23 km/h (TomTom Traffic)
  ✅ Route A->C: 40.0 phút, 15 km/h (TomTom Traffic)
  ✅ Route B->D: 23.076923076923077 phút, 26 km/h (TomTom Traffic)
🍃 Đã thêm 5 bản ghi vào MongoDB
💾 Đã cập nhật 5 bản ghi vào data/traffic_data.csv
Lỗi ở lần lặp 1: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	

...

  ✅ Route B->C: 23.076923076923077 phút, 26 km/h (TomTom Traffic)


127.0.0.1 - - [29/Apr/2026 21:15:43] "POST /api/generate_heatmap HTTP/1.1" 404 -


  ✅ Route C->D: 25.0 phút, 24 km/h (TomTom Traffic)
  ✅ Route A->C: 23.076923076923077 phút, 26 km/h (TomTom Traffic)
  ✅ Route B->D: 23.076923076923077 phút, 26 km/h (TomTom Traffic)
🍃 Đã thêm 5 bản ghi vào MongoDB
💾 Đã cập nhật 5 bản ghi vào data/traffic_data.csv
Lỗi ở lần lặp 3: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1875)
	at org.apache.spark.SparkContext.$anonfun$

127.0.0.1 - - [29/Apr/2026 21:15:54] "POST /api/experiment?iterations=3 HTTP/1.1" 200 -
127.0.0.1 - - [29/Apr/2026 21:15:55] "GET /api/report HTTP/1.1" 200 -


📊 Đánh giá: MAE=0.31, RMSE=1.06, MAPE=1.7%
   So với baseline (MAE=5.40): cải thiện 94.3%
